# Run SceneForge Backend on Kaggle

Runs the FastAPI backend on a Kaggle GPU instance and exposes it via ngrok
so you can hit it from the frontend (running locally or anywhere else).

**Before running:**
1. Enable GPU: Settings → Accelerator → GPU T4 x2 (or P100)
2. Enable Internet: Settings → Internet → On
3. Get a free ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
4. Add it as a Kaggle Secret named `NGROK_TOKEN` (Add-ons → Secrets), or paste it directly below
5. Add your `HF_TOKEN` as a Kaggle Secret too (for gated Llama model access)

## 1 · Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn transformers peft bitsandbytes accelerate \
    pydantic huggingface_hub pyngrok nest_asyncio

## 2 · Load Secrets (HF token + ngrok token)

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# ── Option A: Kaggle Secrets (recommended) ──────────────────────
HF_TOKEN     = secrets.get_secret('HF_TOKEN')
NGROK_TOKEN  = secrets.get_secret('NGROK_TOKEN')

# ── Option B: paste directly (less secure, fine for quick testing) ──
# HF_TOKEN    = 'hf_xxxxxxxxxxxxxxxxxxxx'
# NGROK_TOKEN = 'xxxxxxxxxxxxxxxxxxxxxxxx'

import os
os.environ['HF_TOKEN'] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN)
print('HF login successful.')

## 3 · GPU Check

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

## 4 · Write `main.py` (FastAPI Backend)

In [ ]:
%%writefile /kaggle/working/main.py
"""
main.py — Genre Screenplay Generation API (Kaggle-hosted)
"""

import gc
import time
import torch
from contextlib import asynccontextmanager
from typing import Optional

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ── Config ────────────────────────────────────────────────────────
HF_USERNAME = 'SatyaMoulika'
BASE_MODEL  = 'meta-llama/Llama-3.2-3B-Instruct'

ADAPTERS = {
    'drama':  f'{HF_USERNAME}/llama-3.2-drama-lora',
    'horror': f'{HF_USERNAME}/llama-3.2-horror-lora',
    'scifi':  f'{HF_USERNAME}/llama-3.2-scifi-lora',
    'comedy': f'{HF_USERNAME}/llama-3.2-comedy-lora',
}


class ModelState:
    def __init__(self):
        self.current_genre = None
        self.model = None
        self.tokenizer = None

    def load(self, genre: str):
        if self.current_genre == genre and self.model is not None:
            return
        self._unload()
        print(f'[ModelState] Loading {genre} adapter...')

        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
        )
        tok = AutoTokenizer.from_pretrained(BASE_MODEL)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token

        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, quantization_config=bnb,
            device_map='auto', torch_dtype=torch.float16,
        )
        model = PeftModel.from_pretrained(base, ADAPTERS[genre])
        model.eval()

        self.model, self.tokenizer, self.current_genre = model, tok, genre
        print(f'[ModelState] {genre} adapter ready.')

    def _unload(self):
        if self.model is not None:
            del self.model
            self.model, self.tokenizer, self.current_genre = None, None, None
            gc.collect()
            torch.cuda.empty_cache()

    def generate(self, genre, prompt, max_new_tokens, temperature, top_p):
        self.load(genre)
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True,
                                max_length=512).to(self.model.device)
        t0 = time.perf_counter()
        with torch.no_grad():
            out = self.model.generate(
                **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
                top_p=top_p, do_sample=True, pad_token_id=self.tokenizer.eos_token_id,
            )
        latency_ms = round((time.perf_counter() - t0) * 1000, 1)
        full  = self.tokenizer.decode(out[0], skip_special_tokens=True)
        scene = full.split('### Screenplay:')[-1].strip()
        return scene, latency_ms


model_state = ModelState()


@asynccontextmanager
async def lifespan(app: FastAPI):
    print('API starting up...')
    yield
    model_state._unload()
    print('API shut down.')


app = FastAPI(title='SceneForge API', lifespan=lifespan)
app.add_middleware(CORSMiddleware, allow_origins=['*'],
                   allow_methods=['*'], allow_headers=['*'])


class GenerateRequest(BaseModel):
    genre:          str
    scene_heading:  str
    story_context:  str
    characters:     Optional[str] = None
    tone:           Optional[str] = None
    max_new_tokens: int   = Field(400, ge=50, le=800)
    temperature:    float = Field(0.8, ge=0.1, le=1.5)
    top_p:          float = Field(0.9, ge=0.1, le=1.0)


class GenerateResponse(BaseModel):
    genre: str
    screenplay: str
    latency_ms: float
    prompt_used: str


@app.get('/')
def root():
    return {'message': 'SceneForge API is running.', 'genres': list(ADAPTERS.keys())}


@app.get('/health')
def health():
    return {'status': 'ok', 'loaded_genre': model_state.current_genre,
            'cuda_available': torch.cuda.is_available()}


@app.get('/genres')
def list_genres():
    return {'genres': list(ADAPTERS.keys())}


@app.post('/generate', response_model=GenerateResponse)
def generate(req: GenerateRequest):
    if req.genre not in ADAPTERS:
        raise HTTPException(400, f'Unknown genre. Choose from: {list(ADAPTERS.keys())}')

    extra_lines = []
    if req.characters:
        extra_lines.append(f'Characters: {req.characters}')
    if req.tone:
        extra_lines.append(f'Tone: {req.tone}')
    extras = ('\n' + '\n'.join(extra_lines)) if extra_lines else ''

    prompt = (
        f'### Instruction:\nWrite a {req.genre} screenplay scene.\n'
        f'Scene heading: {req.scene_heading}\n'
        f'Story context: {req.story_context}{extras}\n\n'
        f'### Screenplay:\n'
    )

    try:
        screenplay, latency_ms = model_state.generate(
            req.genre, prompt, req.max_new_tokens, req.temperature, req.top_p)
    except Exception as e:
        raise HTTPException(500, str(e))

    return GenerateResponse(genre=req.genre, screenplay=screenplay,
                            latency_ms=latency_ms, prompt_used=prompt)

## 5 · Start ngrok Tunnel

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)

# Kill any existing tunnels first
ngrok.kill()

public_url = ngrok.connect(8000, bind_tls=True)
print(f'\n🌐 Public API URL: {public_url}')
print(f'   Docs: {public_url}/docs')
print(f'\nUpdate your frontend\'s API_BASE to: {public_url}')

## 6 · Run FastAPI Server

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()

# This cell blocks and keeps the server running.
# Stop the cell (interrupt kernel) to shut the server down.
uvicorn.run('main:app', host='0.0.0.0', port=8000, app_dir='/kaggle/working')

## 7 · Test the API (run in a NEW cell while server cell above is running)

Kaggle notebooks run cells sequentially in one kernel, so the server cell above
will block. To test, open the `public_url` printed in Cell 5 directly in your
browser (append `/docs` for interactive Swagger UI), or run this from your
**local machine** / **frontend** pointed at that URL.

Example curl test (run locally, not in this notebook):
```bash
curl -X POST https://YOUR-NGROK-URL.ngrok-free.app/generate \
  -H "Content-Type: application/json" \
  -d '{
    "genre": "horror",
    "scene_heading": "EXT. ABANDONED FARMHOUSE - NIGHT",
    "story_context": "A group of friends realise they are not alone."
  }'
```